# GIRF - Dyn-method with triangular pulses

Estimate the gradient impulse response function using the Dyn-method with triangular pulses

### Imports

In [ ]:
import tempfile
from pathlib import Path

import MRzeroCore as mr0
from mrpro.data import KData
from mrpro.data.traj_calculators import KTrajectoryCartesian

from mrseq.scripts.girf_dyn_triangle import main as create_seq
from mrseq.utils import sys_defaults

### Settings
We are going to use a numerical phantom with a matrix size of 128 x 128. 

In [ ]:
image_matrix_size = [128, 128]

tmp = tempfile.TemporaryDirectory()
fname_mrd = Path(tmp.name) / 'girf_dyn_triangle.mrd'

### Create the digital phantom

We use the standard Brainweb phantom from [MRzero](https://github.com/MRsources/MRzero-Core).

In [ ]:
phantom = mr0.util.load_phantom(image_matrix_size)

### Create the GIRF Dyn sequence

To create the GIRF Dyn sequence, we use the previously imported [girf_dyn_triangle script](../src/mrseq/scripts/girf_dyn_triangle.py).

In [ ]:
sequence, fname_seq = create_seq(
    system=sys_defaults,
    test_report=False,
    timing_check=False,
)

### Simulate the sequence

Now, we pass the sequence and the phantom to the MRzero simulation and save the simulated signal as an (ISMR)MRD file.

In [ ]:
mr0_sequence = mr0.Sequence.import_file(str(fname_seq.with_suffix('.seq')))
signal, ktraj_adc = mr0.util.simulate(mr0_sequence, phantom, accuracy=1e0)
mr0.sig_to_mrd(fname_mrd, signal, sequence)

### Estimate GIRF

In [ ]:
kdata = KData.from_file(fname_mrd, trajectory=KTrajectoryCartesian())